# Similarity search

Embeddings are dense fixed-length vectors, so cosine distance is a natural proxy for light-curve
similarity.  The example below reads a ZTF DR23 [HATS](https://hats.readthedocs.io) pixel directly
from the public S3 bucket using
[`nested-pandas`](https://nested-pandas.readthedocs.io) and `s3fs`
(install both with `pip install nested-pandas s3fs`), embeds all well-observed light curves with
Astromer2, and finds the closest neighbour to a given object by cosine distance.

This pixel contains ~50 k objects passing the quality cuts; embedding takes roughly
12 minutes on M2 Pro (~15 ms per object).

In [ ]:
# %pip install light-curve huggingface_hub onnxruntime nested-pandas universal-pathlib

In [ ]:
import matplotlib.pyplot as plt
import nested_pandas as npd
import numpy as np
from scipy.spatial.distance import cdist
from upath import UPath

from light_curve.embed import Astromer2

TARGET_OID = 680213300009232  # ZTF r-band light curve
MIN_OBS = 1000

## Step 1 — Load data

Read one HATS pixel of ZTF DR23 directly from the public S3 bucket and keep only
objects with clean photometry (`catflags == 0`) and at least 1 000 observations.

In [ ]:
nf = npd.read_parquet(
    UPath(
        "s3://ipac-irsa-ztf/contributed/dr23/lc/hats/ztf_dr23_lc-hats"
        "/dataset/Norder=5/Dir=0/Npix=2378/",
        anon=True,
    )
)
nf = nf.query("lightcurve.catflags == 0").query(f"lightcurve.list_lengths >= {MIN_OBS}")
print(f"Objects after quality cuts: {len(nf):,}")

## Step 2 — Embed with Astromer2

Load the pretrained model and run it over all light curves with `map_rows`.
Each call returns a 256-dim vector; the results are stored as a nested column
`embedding.value` and then stacked into a matrix for distance computation.

In [ ]:
model = Astromer2.from_hf(output="mean", reduction="beginning")


def embed_row(hmjd, mag):
    return {"embedding.value": model(hmjd, mag).squeeze()}


nf = nf.map_rows(
    embed_row,
    columns=["lightcurve.hmjd", "lightcurve.mag"],
    row_container="args",
    append_columns=True,
)
print(f"Embedded {len(nf):,} objects, embedding dim = {nf['embedding.value'].iloc[0].shape[0]}")

## Step 3 — Find nearest neighbour

Stack the embeddings into a matrix and compute cosine distances from the query object
to all others.  Cosine distance is scale-invariant: only the direction of the embedding
vector matters, not its magnitude.

In [ ]:
oids = nf["objectid"].to_numpy()
matrix = np.asarray(nf["embedding.value"]).reshape(len(nf), -1)

query_idx = np.where(oids == TARGET_OID)[0][0]
distances = cdist(matrix[query_idx : query_idx + 1], matrix, metric="cosine")[0]
distances[query_idx] = np.inf  # exclude the query itself

best_idx = np.argmin(distances)
best_oid = oids[best_idx]
print(f"Query:             OID {TARGET_OID}")
print(f"Nearest neighbour: OID {best_oid}, cosine distance {distances[best_idx]:.6f}")
# Nearest neighbour: OID 680113300005170, cosine distance 0.000046

assert best_oid == 680113300005170
assert distances[best_idx] < 0.001

The nearest neighbour is OID `680113300005170` — the same physical object
([HZ Her / Her X-1](https://en.wikipedia.org/wiki/Hercules_X-1), an X-ray binary) observed in the
*g*-band, recovered automatically from an *r*-band query through embedding similarity.

See both objects on SNAD Viewer:
[query (r-band)](https://ztf.snad.space/dr23/view/680213300009232) ·
[nearest neighbour (g-band)](https://ztf.snad.space/dr23/view/680113300005170)

## Step 4 — Compare light curves

Overlay the two light curves to confirm they show the same periodic variability
(Her X-1 has a ~1.7-day X-ray/optical period).  Note that ZTF uses different
magnitude zero-points per band, so the absolute magnitudes differ while the
variability pattern is the same.

In [ ]:
hmjd_q = nf["lightcurve.hmjd"].iloc[query_idx]
mag_q = nf["lightcurve.mag"].iloc[query_idx]
hmjd_n = nf["lightcurve.hmjd"].iloc[best_idx]
mag_n = nf["lightcurve.mag"].iloc[best_idx]

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(hmjd_q, mag_q, s=3, alpha=0.4, color="tab:red", label=f"Query {TARGET_OID} (r-band)")
ax.scatter(hmjd_n, mag_n, s=3, alpha=0.4, color="tab:blue", label=f"Nearest {best_oid} (g-band)")
ax.invert_yaxis()
ax.set_xlabel("HMJD")
ax.set_ylabel("Magnitude")
ax.set_title("HZ Her / Her X-1 — query (r-band) and nearest neighbour (g-band)")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

## Next steps

- [Classification](classification.ipynb) — train a classifier on top of embeddings
- [onnxruntime tips](../onnxruntime.md) — thread control on shared HPC nodes, GPU/CUDA setup
- [API reference](../api.md)